# Análise Exploratória e Regras de Associação — MovieLens 1M

**Disciplina:** Inteligência Artificial  
**Instituição:** FIAP  
**Dataset:** MovieLens 1M (GroupLens Research)

---

In [ ]:
import os, gc, re, time
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from collections import Counter
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid', palette='deep')
plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['figure.dpi'] = 120
plt.rcParams['axes.titlesize'] = 14
plt.rcParams['axes.labelsize'] = 12

# Criar pasta de outputs
os.makedirs('output', exist_ok=True)

In [ ]:
# ========== CARREGAR DADOS ==========
# Ajuste esses caminhos conforme sua máquina
RATINGS_PATH = 'ratings.dat'
USERS_PATH   = 'users.dat'
MOVIES_PATH  = 'movies.dat'

ratings = pd.read_csv(
    RATINGS_PATH, sep='::', header=None,
    names=['UserID','MovieID','Rating','Timestamp'],
    engine='python', encoding='latin-1'
)
users = pd.read_csv(
    USERS_PATH, sep='::', header=None,
    names=['UserID','Gender','Age','Occupation','Zip'],
    engine='python', encoding='latin-1'
)

HAS_MOVIES = os.path.exists(MOVIES_PATH)
if HAS_MOVIES:
    movies = pd.read_csv(
        MOVIES_PATH, sep='::', header=None,
        names=['MovieID','Title','Genres'],
        engine='python', encoding='latin-1'
    )
    movies['Year'] = movies['Title'].str.extract(r'\((\d{4})\)').astype(float)
    print(f"movies:  {movies.shape}")
else:
    print("⚠️ movies.dat não encontrado — seções 3.3/3.5/4.5 usarão MovieID numérico.")

print(f"ratings: {ratings.shape}")
print(f"users:   {users.shape}")
print("Dados carregados!")

---
## 3.1 — Visão Geral do Dataset
Shape, tipos de dados, valores ausentes, duplicatas e estatísticas descritivas.

In [ ]:
print("=" * 50)
print("RATINGS")
print("=" * 50)
print(f"Shape: {ratings.shape}")
print(f"\nTipos:\n{ratings.dtypes}")
print(f"\nAusentes:\n{ratings.isnull().sum()}")
print(f"Duplicatas: {ratings.duplicated().sum()}")
print(f"\nEstatísticas dos ratings:")
display(ratings['Rating'].describe().to_frame().T)

print("\n" + "=" * 50)
print("USERS")
print("=" * 50)
print(f"Shape: {users.shape}")
print(f"Ausentes:\n{users.isnull().sum()}")
print(f"\nGênero:\n{users['Gender'].value_counts().to_string()}")

if HAS_MOVIES:
    print("\n" + "=" * 50)
    print("MOVIES")
    print("=" * 50)
    print(f"Shape: {movies.shape}")
    print(f"Ausentes:\n{movies.isnull().sum()}")
    display(movies.head(5))

**Observações 3.1:**
- ~1 milhão de avaliações, 6.040 usuários e ~3.700 filmes.
- Sem valores ausentes nem duplicatas — dataset limpo.
- Notas de 1 a 5 (inteiras), média ~3.58 — leve viés positivo (pessoas avaliam mais o que gostam).

---
## 3.2 — Distribuição de Avaliações
Histograma de notas, distribuição por usuário/filme e curva de cauda longa.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

vc = ratings['Rating'].value_counts().sort_index()
vc.plot(kind='bar', ax=axes[0], color=sns.color_palette('deep'), edgecolor='black')
axes[0].set_title('Distribuição das Notas')
axes[0].set_xlabel('Nota'); axes[0].set_ylabel('Frequência')
for p in axes[0].patches:
    axes[0].annotate(f'{int(p.get_height()):,}', (p.get_x()+p.get_width()/2., p.get_height()),
                     ha='center', va='bottom', fontsize=9)

pct = ratings['Rating'].value_counts(normalize=True).sort_index()*100
pct.plot(kind='bar', ax=axes[1], color=sns.color_palette('muted'), edgecolor='black')
axes[1].set_title('Proporção das Notas (%)')
axes[1].set_xlabel('Nota'); axes[1].set_ylabel('%')
for p in axes[1].patches:
    axes[1].annotate(f'{p.get_height():.1f}%', (p.get_x()+p.get_width()/2., p.get_height()),
                     ha='center', va='bottom', fontsize=9)
plt.tight_layout(); plt.savefig('output/3_2_notas.png', bbox_inches='tight'); plt.show()

In [ ]:
rpu = ratings.groupby('UserID').size()
rpm = ratings.groupby('MovieID').size()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].hist(rpu, bins=50, color='steelblue', edgecolor='black', alpha=0.8)
axes[0].set_title('Avaliações por Usuário')
axes[0].set_xlabel('Nº de avaliações'); axes[0].set_ylabel('Nº de usuários')
axes[0].axvline(rpu.median(), color='red', ls='--', label=f'Mediana={rpu.median():.0f}')
axes[0].legend()

axes[1].hist(rpm, bins=50, color='coral', edgecolor='black', alpha=0.8)
axes[1].set_title('Avaliações por Filme')
axes[1].set_xlabel('Nº de avaliações'); axes[1].set_ylabel('Nº de filmes')
axes[1].axvline(rpm.median(), color='red', ls='--', label=f'Mediana={rpm.median():.0f}')
axes[1].legend()
plt.tight_layout(); plt.savefig('output/3_2_dist_user_movie.png', bbox_inches='tight'); plt.show()

print(f"Avaliações/usuário: min={rpu.min()}, max={rpu.max()}, média={rpu.mean():.1f}, mediana={rpu.median():.0f}")
print(f"Avaliações/filme:   min={rpm.min()}, max={rpm.max()}, média={rpm.mean():.1f}, mediana={rpm.median():.0f}")

In [ ]:
pop = rpm.sort_values(ascending=False).reset_index(drop=True)
fig, ax = plt.subplots(figsize=(14, 5))
ax.fill_between(range(len(pop)), pop.values, alpha=0.4, color='steelblue')
ax.plot(pop.values, color='steelblue', lw=1.5)
ax.set_title('Curva de Cauda Longa — Filmes por nº de avaliações')
ax.set_xlabel('Filmes (rank)'); ax.set_ylabel('Nº de avaliações')
cumsum = pop.cumsum(); total = pop.sum()
idx80 = (cumsum >= total*0.80).idxmax()
ax.axvline(idx80, color='red', ls='--',
           label=f'80% avaliações nos top {idx80} filmes ({idx80/len(pop)*100:.1f}% do catálogo)')
ax.legend(fontsize=10)
plt.tight_layout(); plt.savefig('output/3_2_long_tail.png', bbox_inches='tight'); plt.show()
print(f"80% das avaliações em {idx80} filmes = {idx80/len(pop)*100:.1f}% do catálogo")

**Observações 3.2:**
- Nota **4** é a mais frequente (~34%), seguida de 3 (~24%). Notas 1-2 são minoria — viés de seleção.
- Distribuição por usuário é assimétrica: maioria avalia poucos filmes, mas há super-avaliadores com 1000+.
- **Cauda longa:** ~20% dos filmes concentram ~80% das avaliações (lei de potência). Filmes da "cauda" têm suporte baixíssimo e não aparecerão nas regras de associação.

---
## 3.3 — Filmes Mais Avaliados
Top-20 por volume e por nota média (com mínimo de 100 avaliações).

In [ ]:
ms = ratings.groupby('MovieID').agg(
    n_ratings=('Rating','count'), mean_rating=('Rating','mean')
).reset_index()

if HAS_MOVIES:
    ms = ms.merge(movies[['MovieID','Title','Genres']], on='MovieID', how='left')
    lbl = 'Title'
else:
    lbl = 'MovieID'

top20c = ms.nlargest(20, 'n_ratings')
fig, ax = plt.subplots(figsize=(12, 8))
ax.barh(range(20), top20c['n_ratings'].values, color=sns.color_palette('viridis',20), edgecolor='black')
ax.set_yticks(range(20)); ax.set_yticklabels(top20c[lbl].values, fontsize=9)
ax.invert_yaxis(); ax.set_xlabel('Nº de Avaliações')
ax.set_title('Top-20 Filmes por Nº de Avaliações')
for i, v in enumerate(top20c['n_ratings'].values):
    ax.text(v+10, i, f'{v:,}', va='center', fontsize=8)
plt.tight_layout(); plt.savefig('output/3_3_top20_count.png', bbox_inches='tight'); plt.show()

qual = ms[ms['n_ratings'] >= 100]
top20m = qual.nlargest(20, 'mean_rating')
fig, ax = plt.subplots(figsize=(12, 8))
ax.barh(range(20), top20m['mean_rating'].values, color=sns.color_palette('YlOrRd_r',20), edgecolor='black')
ax.set_yticks(range(20)); ax.set_yticklabels(top20m[lbl].values, fontsize=9)
ax.invert_yaxis(); ax.set_xlabel('Média de Nota')
ax.set_title('Top-20 Filmes por Média de Nota (mín. 100 avaliações)')
ax.set_xlim(3.5, 5.0)
for i, v in enumerate(top20m['mean_rating'].values):
    ax.text(v+0.02, i, f'{v:.2f}', va='center', fontsize=8)
plt.tight_layout(); plt.savefig('output/3_3_top20_mean.png', bbox_inches='tight'); plt.show()

**Observações 3.3:**
- Os mais avaliados são blockbusters universais (Star Wars, Matrix, Pulp Fiction, etc.).
- No ranking por nota, clássicos e filmes cult com público fiel dominam.
- **Viés de popularidade:** regras de associação serão dominadas por filmes populares — regras entre blockbusters são "óbvias" e pouco úteis para recomendação.

---
## 3.4 — Perfil dos Usuários
Segmentação por intensidade de avaliação, gênero e idade.

In [ ]:
us = ratings.groupby('UserID').agg(n=('Rating','count'), mean_r=('Rating','mean')).reset_index()
us = us.merge(users, on='UserID')

def perfil(n):
    if n <= 50: return 'Esporádico (≤50)'
    elif n <= 150: return 'Regular (51-150)'
    elif n <= 500: return 'Ativo (151-500)'
    else: return 'Super-ativo (500+)'

us['Perfil'] = us['n'].apply(perfil)
order = ['Esporádico (≤50)', 'Regular (51-150)', 'Ativo (151-500)', 'Super-ativo (500+)']

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
us['Perfil'].value_counts().reindex(order).plot(
    kind='bar', ax=axes[0], color=['#2196F3','#4CAF50','#FF9800','#F44336'], edgecolor='black')
axes[0].set_title('Perfis de Usuário'); axes[0].set_ylabel('Nº de Usuários')
axes[0].tick_params(axis='x', rotation=30)
for p in axes[0].patches:
    axes[0].annotate(f'{int(p.get_height()):,}', (p.get_x()+p.get_width()/2., p.get_height()),
                     ha='center', va='bottom', fontsize=9)

us.groupby('Gender')['n'].mean().plot(kind='bar', ax=axes[1], color=['#E91E63','#2196F3'], edgecolor='black')
axes[1].set_title('Média de Avaliações por Gênero'); axes[1].set_ylabel('Média'); axes[1].tick_params(axis='x', rotation=0)
plt.tight_layout(); plt.savefig('output/3_4_perfil.png', bbox_inches='tight'); plt.show()

for p in order:
    sub = us[us['Perfil']==p]
    print(f"{p}: {len(sub)} ({len(sub)/len(us)*100:.1f}%), avg_filmes={sub['n'].mean():.0f}, avg_nota={sub['mean_r'].mean():.2f}")

**Observações 3.4:**
- Maioria é esporádica (poucas avaliações), mas super-ativos (500+) geram parcela significativa do dataset.
- Homens avaliam mais filmes em média que mulheres (viés demográfico do dataset).
- Para regras de associação, super-avaliadores influenciam desproporcionalmente o suporte dos itemsets.

---
## 3.5 — Análise por Gênero
Distribuição de filmes e avaliações por gênero cinematográfico.

In [ ]:
if HAS_MOVIES:
    ge = movies.assign(Genre=movies['Genres'].str.split('|')).explode('Genre')
    gr = ge.merge(ratings, on='MovieID')

    fig, axes = plt.subplots(1, 2, figsize=(16, 7))
    ge['Genre'].value_counts().plot(kind='barh', ax=axes[0], color='steelblue', edgecolor='black')
    axes[0].set_title('Nº de Filmes por Gênero'); axes[0].set_xlabel('Filmes'); axes[0].invert_yaxis()

    gm = gr.groupby('Genre')['Rating'].mean().sort_values()
    colors = plt.cm.RdYlGn(np.linspace(0.2, 0.8, len(gm)))
    gm.plot(kind='barh', ax=axes[1], color=colors, edgecolor='black')
    axes[1].set_title('Média de Nota por Gênero'); axes[1].set_xlabel('Média'); axes[1].set_xlim(2.5,4.5)
    axes[1].invert_yaxis()
    plt.tight_layout(); plt.savefig('output/3_5_generos.png', bbox_inches='tight'); plt.show()

    gs = gr.groupby('Genre').agg(n_filmes=('MovieID','nunique'), n_aval=('Rating','count'), media=('Rating','mean'))
    print(gs.sort_values('n_aval', ascending=False).to_string())
else:
    print("⚠️ movies.dat necessário para análise por gênero.")

**Observações 3.5:**
- Drama e Comédia dominam em volume. Film-Noir e Documentário têm médias altas mas amostra pequena (viés de nicho).
- Horror e Children's têm menores médias — gêneros polarizantes.
- Gêneros dominantes (Drama/Comedy/Action) aparecem em quase toda regra como "ruído" — regras mais úteis viriam de subgêneros específicos.

---
## 3.6 — Esparsidade da Matriz
Cálculo e interpretação da esparsidade da matriz usuário × filme.

In [ ]:
n_u = ratings['UserID'].nunique()
n_m = ratings['MovieID'].nunique()
n_r = len(ratings)
n_pos = n_u * n_m
sparsity = 1 - (n_r / n_pos)

print(f"Usuários:    {n_u:,}")
print(f"Filmes:      {n_m:,}")
print(f"Avaliações:  {n_r:,}")
print(f"Posições:    {n_pos:,}")
print(f"Preenchidas: {n_r/n_pos*100:.2f}%")
print(f"\n{'═'*40}")
print(f"  ESPARSIDADE = {sparsity*100:.2f}%")
print(f"{'═'*40}")
print(f"\nCada usuário avaliou ~{n_r/n_u:.0f} de {n_m:,} filmes em média.")

np.random.seed(42)
su = np.random.choice(ratings['UserID'].unique(), 200, replace=False)
sm = np.random.choice(ratings['MovieID'].unique(), 300, replace=False)
samp = ratings[ratings['UserID'].isin(su) & ratings['MovieID'].isin(sm)]
mat = samp.pivot_table(index='UserID', columns='MovieID', values='Rating')

fig, ax = plt.subplots(figsize=(14, 6))
ax.spy(mat.notna().values, markersize=0.5, aspect='auto', color='steelblue')
ax.set_title(f'Amostra da Matriz Usuário×Filme (200×300) — Esparsidade: {sparsity*100:.2f}%')
ax.set_xlabel('Filmes'); ax.set_ylabel('Usuários')
plt.tight_layout(); plt.savefig('output/3_6_esparsidade.png', bbox_inches='tight'); plt.show()

**Observações 3.6:**
- Esparsidade de **~95.5%** — apenas ~4.5% das posições preenchidas.
- Com suporte de 10%, exigimos que ≥604 dos 6.040 usuários tenham curtido ambos os filmes — restringe regras a blockbusters.
- **Relação direta:** esparsidade alta → suporte mínimo baixo necessário → mas suporte muito baixo → regras espúrias. Solução: suporte moderado + filtro por lift e confiança altos.

---
## 4.1 — Transformação para Formato Transacional
Cada usuário = transação. Itens = filmes com nota ≥ 4 (binarização por "gostou").

In [ ]:
THRESHOLD = 4
liked = ratings[ratings['Rating'] >= THRESHOLD].copy()
print(f"Avaliações totais: {len(ratings):,}")
print(f"Com nota >= {THRESHOLD}: {len(liked):,} ({len(liked)/len(ratings)*100:.1f}%)")

transactions = liked.groupby('UserID')['MovieID'].apply(list).reset_index()
transactions.columns = ['UserID', 'Movies']
transactions['n_items'] = transactions['Movies'].apply(len)

print(f"Transações: {len(transactions):,}")
print(f"Itens/transação: min={transactions['n_items'].min()}, max={transactions['n_items'].max()}, "
      f"média={transactions['n_items'].mean():.1f}, mediana={transactions['n_items'].median():.0f}")

fig, ax = plt.subplots(figsize=(12, 4))
ax.hist(transactions['n_items'], bins=50, color='steelblue', edgecolor='black', alpha=0.8)
ax.set_title('Tamanho das Transações (filmes curtidos por usuário)')
ax.set_xlabel('Nº de filmes (nota ≥ 4)'); ax.set_ylabel('Nº de usuários')
ax.axvline(transactions['n_items'].median(), color='red', ls='--',
           label=f'Mediana={transactions["n_items"].median():.0f}')
ax.legend()
plt.tight_layout(); plt.savefig('output/4_1_transacoes.png', bbox_inches='tight'); plt.show()

In [ ]:
from mlxtend.preprocessing import TransactionEncoder

MIN_SUP_GLOBAL = 0.05
min_count_global = int(MIN_SUP_GLOBAL * len(transactions))

movie_freq = liked.groupby('MovieID').size()
frequent_movies = set(movie_freq[movie_freq >= min_count_global].index)
print(f"Filmes com suporte >= {MIN_SUP_GLOBAL}: {len(frequent_movies)} de {liked['MovieID'].nunique()}")

filtered_transactions = transactions['Movies'].apply(
    lambda x: [str(m) for m in x if m in frequent_movies]
).tolist()
filtered_transactions = [t for t in filtered_transactions if len(t) > 0]
print(f"Transações após filtro: {len(filtered_transactions)}")

te = TransactionEncoder()
te_array = te.fit(filtered_transactions).transform(filtered_transactions)
basket = pd.DataFrame(te_array, columns=te.columns_)
print(f"Matriz binária: {basket.shape[0]} × {basket.shape[1]}")
print(f"Esparsidade transacional: {(1 - basket.values.sum()/(basket.shape[0]*basket.shape[1]))*100:.2f}%")
gc.collect()

**Observações 4.1:**
- Threshold de **nota ≥ 4** para binarizar: nota 4-5 = "gostou" (item na transação). Notas 1-3 descartadas.
- Justificativa (da EDA 3.2): notas 4+5 = ~55% das avaliações, volume suficiente.
- Pré-filtragem: removemos filmes que não atingem suporte mínimo global (5%) antes de montar a matriz. Isso reduz a dimensionalidade sem perder nenhuma regra que seria encontrada.

---
## 4.2 — Escolha e Justificativa do Algoritmo

### Algoritmo: **FP-Growth**

| Critério | Apriori | FP-Growth |
|---|---|---|
| Candidatos | Gera e testa todos (2^n pior caso) | Nenhum — usa FP-Tree |
| Passagens | Múltiplas (1 por tamanho de itemset) | Apenas 2 |
| Performance | Lento com ~1M ratings | Muito mais rápido |
| Memória | Pode explodir | Controlada pela compressão |

**Por que FP-Growth?**
1. **Tamanho (1M ratings, 6K users, 3.7K filmes):** Apriori geraria milhões de candidatos descartados.
2. **Esparsidade (~96%):** maioria das combinações nunca co-ocorre — FP-Tree poda automaticamente.
3. **Cauda longa:** FP-Growth ignora itens infrequentes na construção da árvore.

---
## 4.3 — Definição de Hiperparâmetros

In [ ]:
n_trans = basket.shape[0]
movie_freq_trans = basket.sum()

print(f"Total de transações: {n_trans}")
print()
for sup in [0.25, 0.20, 0.15, 0.10, 0.07, 0.05]:
    mc = int(sup * n_trans)
    nq = (movie_freq_trans >= mc).sum()
    print(f"Suporte {sup*100:5.1f}%  →  mín {mc:,} transações  →  {nq} filmes qualificam")

print()
print("═" * 50)
print("HIPERPARÂMETROS ESCOLHIDOS:")
print("  Suporte mínimo:   0.07 (7%)")
print("  Confiança mínima: 0.50 (50%)")
print("  Lift mínimo:      1.0 (filtro pós-geração)")
print("═" * 50)

**Justificativa:**
- **Suporte = 7%:** balanceia entre ter regras suficientes (~200-400 filmes qualificados) e evitar explosão combinatória.
- **Confiança = 50%:** "se gostou de A, em ≥50% dos casos gostou de B" — threshold mínimo para recomendação acionável.
- **Lift > 1:** filtra associações que ocorrem **mais** que o acaso. Priorizamos lift > 2 (associação forte).

---
## 4.4 — Extração das Regras (Preferência + Aversão)
FP-Growth em duas matrizes: filmes **curtidos** (nota ≥ 4) e filmes **rejeitados** (nota ≤ 2).

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# FUNÇÕES AUXILIARES — Filtro de Regras Óbvias
# ─────────────────────────────────────────────────────────────────────────────

# Palavras ignoradas na comparação de títulos
_STOPWORDS = {
    'the','a','an','of','and','in','to','with','at','by','for','on','is','it',
    'as','or','de','da','do','part','episode','vol','volume','chapter'
}
_NUMEROS = re.compile(r'\b(i{1,3}|iv|vi{0,3}|ix|x|[0-9]+)(st|nd|rd|th)?\b', re.IGNORECASE)

def _palavras_chave(titulo: str) -> frozenset:
    """
    Extrai palavras significativas do título:
    remove ano, números romanos/arábicos, pontuação e stopwords.
    Ex: 'The Godfather: Part II (1974)' → frozenset({'godfather'})
        'Star Wars: Episode V (1980)'   → frozenset({'star', 'wars'})
    """
    t = re.sub(r'\(\d{4}\)', '', titulo)    # remove (ano)
    t = re.sub(r'[:\-]', ' ', t)            # : e - viram espaço
    t = _NUMEROS.sub('', t)                 # remove romanos e arábicos
    palavras = re.sub(r'[^a-z\s]', '', t.lower()).split()
    return frozenset(w for w in palavras if w not in _STOPWORDS and len(w) > 2)


def e_regra_obvia(antecedente: str, consequente: str) -> bool:
    """
    Retorna True se ≥60% das palavras-chave forem comuns entre os títulos.
    Detecta sequências, partes e episódios da mesma franquia.

    Exemplos REMOVIDOS:  Godfather → Godfather: Part II  (100% overlap)
                         Star Wars IV → Star Wars V      (100% overlap)
                         Indiana Jones I → Indiana Jones II  (67%)
    Exemplos MANTIDOS:   Pulp Fiction → Forrest Gump     (0%)
                         Silence of the Lambs → Hannibal (0%)
    """
    if antecedente == consequente:
        return True
    pa = _palavras_chave(antecedente)
    pc = _palavras_chave(consequente)
    if not pa or not pc:
        return False
    sobreposicao = len(pa & pc) / min(len(pa), len(pc))
    return sobreposicao >= 0.6


def mapear_titulos(df_rules, id2title_map, has_movies):
    """Adiciona colunas Antecedente e Consequente com títulos legíveis."""
    if has_movies:
        df_rules['Antecedente'] = df_rules['antecedents'].apply(
            lambda x: ', '.join([id2title_map.get(i, i) for i in x]))
        df_rules['Consequente'] = df_rules['consequents'].apply(
            lambda x: ', '.join([id2title_map.get(i, i) for i in x]))
    else:
        df_rules['Antecedente'] = df_rules['antecedents'].apply(lambda x: ', '.join(sorted(x)))
        df_rules['Consequente'] = df_rules['consequents'].apply(lambda x: ', '.join(sorted(x)))
    return df_rules


def filtrar_obvias(df_rules, has_movies):
    """Remove regras óbvias (mesma franquia/série). Só funciona com HAS_MOVIES=True."""
    if not has_movies:
        return df_rules  # sem títulos não é possível comparar
    mascara = df_rules.apply(
        lambda r: e_regra_obvia(r['Antecedente'], r['Consequente']), axis=1
    )
    n_removidas = mascara.sum()
    print(f"  Regras óbvias removidas: {n_removidas} | Mantidas: {(~mascara).sum()}")
    return df_rules[~mascara].copy()


print("✅ Funções auxiliares definidas.")

In [ ]:
from mlxtend.frequent_patterns import fpgrowth, association_rules

# ── Dicionário de títulos ──────────────────────────────────────────────────
if HAS_MOVIES:
    id2title = dict(zip(movies['MovieID'].astype(str), movies['Title']))
else:
    id2title = {}

# ── PARTE A: Regras de PREFERÊNCIA (nota ≥ 4) ─────────────────────────────
MIN_SUPPORT    = 0.07
MIN_CONFIDENCE = 0.50

print("=" * 60)
print("PARTE A — REGRAS DE PREFERÊNCIA (nota ≥ 4)")
print("=" * 60)
print(f"Rodando FP-Growth (suporte ≥ {MIN_SUPPORT})...")
t0 = time.time()
freq_items = fpgrowth(basket, min_support=MIN_SUPPORT, use_colnames=True)
print(f"  {len(freq_items)} itemsets em {time.time()-t0:.1f}s")
freq_items['length'] = freq_items['itemsets'].apply(len)
print(f"  Por tamanho: {freq_items['length'].value_counts().sort_index().to_dict()}")

rules_pref = association_rules(freq_items, metric='confidence', min_threshold=MIN_CONFIDENCE)
rules_pref = rules_pref[rules_pref['lift'] > 1.0].copy()
print(f"  Regras com lift > 1: {len(rules_pref)}")

rules_pref = mapear_titulos(rules_pref, id2title, HAS_MOVIES)
rules_pref['tipo_regra'] = 'preferencia'

print("\nFiltrando regras óbvias (sequências/franquias)...")
rules_pref = filtrar_obvias(rules_pref, HAS_MOVIES)
gc.collect()

In [ ]:
# ── PARTE B: Regras de AVERSÃO (nota ≤ 2) ─────────────────────────────────
THRESHOLD_RUIM  = 2
MIN_SUP_RUIM    = 0.03   # suporte mais baixo — quem odeia filmes é minoria
MIN_CONF_RUIM   = 0.40

print("=" * 60)
print("PARTE B — REGRAS DE AVERSÃO (nota ≤ 2)")
print("=" * 60)

detestou = ratings[ratings['Rating'] <= THRESHOLD_RUIM].copy()
print(f"Avaliações com nota ≤ {THRESHOLD_RUIM}: {len(detestou):,} ({len(detestou)/len(ratings)*100:.1f}%)")

trans_ruins = detestou.groupby('UserID')['MovieID'].apply(list).reset_index()
trans_ruins.columns = ['UserID', 'Movies']
print(f"Usuários que deram nota ≤ {THRESHOLD_RUIM} em ≥ 1 filme: {len(trans_ruins):,}")

min_count_ruim = int(MIN_SUP_RUIM * len(trans_ruins))
freq_ruins = detestou.groupby('MovieID').size()
filmes_freq_ruins = set(freq_ruins[freq_ruins >= min_count_ruim].index)
print(f"Filmes com suporte >= {MIN_SUP_RUIM} nas avaliações ruins: {len(filmes_freq_ruins)}")

rules_aversao = pd.DataFrame()  # fallback

if len(filmes_freq_ruins) >= 2:
    trans_ruins_filtradas = trans_ruins['Movies'].apply(
        lambda x: [str(m) for m in x if m in filmes_freq_ruins]
    ).tolist()
    trans_ruins_filtradas = [t for t in trans_ruins_filtradas if len(t) > 0]

    te_ruim = TransactionEncoder()
    basket_ruim = pd.DataFrame(
        te_ruim.fit(trans_ruins_filtradas).transform(trans_ruins_filtradas),
        columns=te_ruim.columns_
    )
    print(f"Matriz (aversão): {basket_ruim.shape[0]} usuários × {basket_ruim.shape[1]} filmes")

    print(f"Rodando FP-Growth (suporte ≥ {MIN_SUP_RUIM})...")
    t0 = time.time()
    freq_items_ruim = fpgrowth(basket_ruim, min_support=MIN_SUP_RUIM, use_colnames=True)
    print(f"  {len(freq_items_ruim)} itemsets em {time.time()-t0:.1f}s")

    if len(freq_items_ruim) > 0:
        rules_aversao = association_rules(freq_items_ruim, metric='confidence', min_threshold=MIN_CONF_RUIM)
        rules_aversao = rules_aversao[rules_aversao['lift'] > 1.0].copy()
        print(f"  Regras de aversão com lift > 1: {len(rules_aversao)}")

        rules_aversao = mapear_titulos(rules_aversao, id2title, HAS_MOVIES)
        rules_aversao['tipo_regra'] = 'aversao'

        print("\nFiltrando regras óbvias (aversão)...")
        rules_aversao = filtrar_obvias(rules_aversao, HAS_MOVIES)
    else:
        print("Nenhum itemset frequente encontrado nas avaliações ruins.")
else:
    print("⚠️  Poucos filmes frequentes. Tente reduzir MIN_SUP_RUIM para 0.01.")

gc.collect()

In [ ]:
# ── Top-20 por lift — PREFERÊNCIA ─────────────────────────────────────────
top20_pref = rules_pref.nlargest(20, 'lift')
print("═" * 80)
print("TOP-20 REGRAS DE PREFERÊNCIA NÃO-ÓBVIAS (por lift)")
print("═" * 80)
for i, (_, r) in enumerate(top20_pref.iterrows(), 1):
    print(f"\n#{i:2d}  {r['Antecedente']}  →  {r['Consequente']}")
    print(f"     Suporte={r['support']:.4f}  Confiança={r['confidence']:.2%}  Lift={r['lift']:.2f}")

# Tabela formatada
cols = ['Antecedente','Consequente','support','confidence','lift']
t20 = top20_pref[cols].copy().reset_index(drop=True)
t20.index = range(1, len(t20)+1)
t20.columns = ['Antecedente','Consequente','Suporte','Confiança','Lift']
display(t20.style.format({'Suporte':'{:.4f}','Confiança':'{:.2%}','Lift':'{:.2f}'})
        .background_gradient(subset=['Lift'], cmap='YlOrRd'))

In [ ]:
# ── Top-20 por lift — AVERSÃO ──────────────────────────────────────────────
if len(rules_aversao) > 0:
    top20_av = rules_aversao.nlargest(20, 'lift')
    print("═" * 80)
    print("TOP-20 REGRAS DE AVERSÃO NÃO-ÓBVIAS (por lift)")
    print("Leitura: quem NÃO gostou de A tende a NÃO gostar de B")
    print("═" * 80)
    for i, (_, r) in enumerate(top20_av.iterrows(), 1):
        print(f"\n#{i:2d}  {r['Antecedente']}  →  {r['Consequente']}")
        print(f"     Suporte={r['support']:.4f}  Confiança={r['confidence']:.2%}  Lift={r['lift']:.2f}")

    cols = ['Antecedente','Consequente','support','confidence','lift']
    t20a = top20_av[cols].copy().reset_index(drop=True)
    t20a.index = range(1, len(t20a)+1)
    t20a.columns = ['Antecedente','Consequente','Suporte','Confiança','Lift']
    display(t20a.style.format({'Suporte':'{:.4f}','Confiança':'{:.2%}','Lift':'{:.2f}'})
            .background_gradient(subset=['Lift'], cmap='Blues'))
else:
    print("Nenhuma regra de aversão encontrada com os parâmetros atuais.")

In [ ]:
# ── Scatter: Suporte vs Confiança, cor = Lift (ambos os tipos) ─────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

sc = axes[0].scatter(rules_pref['support'], rules_pref['confidence'],
                c=rules_pref['lift'], cmap='YlOrRd', alpha=0.6, s=50, edgecolors='gray', lw=0.5)
plt.colorbar(sc, ax=axes[0], label='Lift')
axes[0].set_xlabel('Suporte'); axes[0].set_ylabel('Confiança')
axes[0].set_title('Preferência — Suporte vs Confiança (cor = Lift)')

if len(rules_aversao) > 0:
    sc2 = axes[1].scatter(rules_aversao['support'], rules_aversao['confidence'],
                    c=rules_aversao['lift'], cmap='Blues', alpha=0.6, s=50, edgecolors='gray', lw=0.5)
    plt.colorbar(sc2, ax=axes[1], label='Lift')
    axes[1].set_xlabel('Suporte'); axes[1].set_ylabel('Confiança')
    axes[1].set_title('Aversão — Suporte vs Confiança (cor = Lift)')
else:
    axes[1].text(0.5, 0.5, 'Sem regras de aversão', ha='center', va='center', transform=axes[1].transAxes)

plt.tight_layout()
plt.savefig('output/4_4_scatter.png', bbox_inches='tight')
plt.show()

---
## 4.5 — Interpretação das Regras
Análise em linguagem natural das regras mais fortes (preferência e aversão).

In [ ]:
print("INTERPRETAÇÃO DAS TOP-5 REGRAS DE PREFERÊNCIA")
print("=" * 60)
for i, (_, r) in enumerate(top20_pref.head(5).iterrows(), 1):
    ant, cons = r['Antecedente'], r['Consequente']
    print(f"\nRegra #{i}: {ant}  →  {cons}")
    print(f"  {r['confidence']:.0%} dos usuários que gostaram de [{ant}]")
    print(f"  também gostaram de [{cons}].")
    print(f"  Lift={r['lift']:.2f}: a chance é {r['lift']:.1f}x maior que no acaso.")

if len(rules_aversao) > 0:
    print("\n" + "=" * 60)
    print("INTERPRETAÇÃO DAS TOP-5 REGRAS DE AVERSÃO")
    print("=" * 60)
    for i, (_, r) in enumerate(rules_aversao.nlargest(5, 'lift').iterrows(), 1):
        ant, cons = r['Antecedente'], r['Consequente']
        print(f"\nRegra #{i}: {ant}  →  {cons}")
        print(f"  {r['confidence']:.0%} dos usuários que NÃO gostaram de [{ant}]")
        print(f"  também NÃO gostaram de [{cons}].")
        print(f"  Lift={r['lift']:.2f}: essa aversão conjunta é {r['lift']:.1f}x mais comum que o acaso.")

print("\n" + "─" * 60)
print("PADRÕES IDENTIFICADOS:")
print("─" * 60)
print("• Regras óbvias (sequências) foram removidas — resultados refletem associações reais de gosto.")
print("• Preferência: filmes do mesmo gênero/diretor/época tendem a co-ocorrer.")
if len(rules_aversao) > 0:
    print("• Aversão: perfis de usuários que rejeitam determinados gêneros compartilham padrões.")
print("\nLIMITAÇÕES:")
print("• Viés de popularidade ainda presente — blockbusters dominam o suporte.")
print("• Aversão tem suporte naturalmente menor — usuários tendem a não avaliar o que não assistiram.")

---
## 4.6 — Análise de Sensibilidade
Impacto da variação do suporte mínimo no número e qualidade das regras.

In [ ]:
support_values = [0.07, 0.10, 0.15, 0.20, 0.25]
results = []

print(f"{'Suporte':>8} {'Itemsets':>10} {'Regras':>8} {'Lift>1':>8} {'Lift médio':>10} {'Lift max':>10} {'Conf média':>10}")
print("─" * 75)

for sup in support_values:
    try:
        fi = fpgrowth(basket, min_support=sup, use_colnames=True)
        fi['length'] = fi['itemsets'].apply(len)
        fi2 = fi[fi['length'] >= 2]
        if len(fi2) > 0:
            ar = association_rules(fi, metric='confidence', min_threshold=0.3)
            arlift = ar[ar['lift'] > 1.0]
            nr, nl = len(ar), len(arlift)
            ml = arlift['lift'].mean() if nl > 0 else 0
            xl = arlift['lift'].max() if nl > 0 else 0
            mc = arlift['confidence'].mean() if nl > 0 else 0
        else:
            nr = nl = 0; ml = xl = mc = 0
        results.append(dict(support=sup, n_itemsets=len(fi), n_rules=nr,
                            n_lift=nl, mean_lift=ml, max_lift=xl, mean_conf=mc))
        print(f"{sup:>8.2f} {len(fi):>10} {nr:>8} {nl:>8} {ml:>10.2f} {xl:>10.2f} {mc:>10.2%}")
    except Exception as e:
        print(f"{sup:>8.2f}  ERRO: {e}")
        results.append(dict(support=sup, n_itemsets=0, n_rules=0, n_lift=0,
                            mean_lift=0, max_lift=0, mean_conf=0))

sens = pd.DataFrame(results)
gc.collect()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

axes[0].plot(sens['support'], sens['n_rules'], 'o-', color='steelblue', lw=2, ms=8)
axes[0].plot(sens['support'], sens['n_lift'], 's--', color='coral', lw=2, ms=8)
axes[0].set_xlabel('Suporte Mínimo'); axes[0].set_ylabel('Nº de Regras')
axes[0].set_title('Quantidade de Regras vs Suporte'); axes[0].legend(['Todas','Lift>1'])
if sens['n_rules'].max() > 100:
    axes[0].set_yscale('log')

axes[1].plot(sens['support'], sens['mean_lift'], 'o-', color='green', lw=2, ms=8)
axes[1].fill_between(sens['support'], sens['mean_lift'], alpha=0.2, color='green')
axes[1].set_xlabel('Suporte Mínimo'); axes[1].set_ylabel('Lift Médio')
axes[1].set_title('Qualidade (Lift Médio) vs Suporte')

axes[2].plot(sens['support'], sens['mean_conf'], 'o-', color='purple', lw=2, ms=8)
axes[2].fill_between(sens['support'], sens['mean_conf'], alpha=0.2, color='purple')
axes[2].set_xlabel('Suporte Mínimo'); axes[2].set_ylabel('Confiança Média')
axes[2].set_title('Confiança Média vs Suporte')
axes[2].yaxis.set_major_formatter(mticker.PercentFormatter(1.0))

plt.tight_layout(); plt.savefig('output/4_6_sensibilidade.png', bbox_inches='tight'); plt.show()

**Observações 4.6:**
- **Trade-off claro:** suporte baixo → muitas regras (risco de espúrias) vs. suporte alto → poucas regras (só blockbusters).
- **Suporte 7%** é o ponto ótimo: quantidade gerenciável de regras com lift e confiança razoáveis.
- Suporte ≥ 20%: poucas regras, todas envolvendo top-50 filmes. Alta confiança, mas regras óbvias.
- Lift médio tende a subir com suporte alto — os poucos filmes que sobrevivem ao filtro têm co-ocorrência forte.

---
## 4.7 — Exportação: Top 50 Regras Não-Óbvias
Combina regras de preferência e aversão, rankeadas por score composto, e exporta para CSV.

In [ ]:
print("=" * 60)
print("4.7 — EXPORTAÇÃO TOP 50 REGRAS NÃO-ÓBVIAS → CSV")
print("=" * 60)

# Colunas base disponíveis
cols_base = ['Antecedente', 'Consequente', 'tipo_regra',
             'support', 'confidence', 'lift', 'leverage', 'conviction', 'zhangs_metric']

def preparar_df(df):
    cols = [c for c in cols_base if c in df.columns]
    return df[cols].copy()

frames = [preparar_df(rules_pref)]
if len(rules_aversao) > 0:
    frames.append(preparar_df(rules_aversao))

df_todas = pd.concat(frames, ignore_index=True)

# Score composto: lift × confiança × log(suporte*100 + 1)
# Garante equilíbrio entre força da regra e suporte real
df_todas['score'] = (
    df_todas['lift'] *
    df_todas['confidence'] *
    np.log1p(df_todas['support'] * 100)
)

# Selecionar top 50 garantindo diversidade de tipo (40 pref + 10 aversão, se disponível)
top_pref = df_todas[df_todas['tipo_regra'] == 'preferencia'].nlargest(40, 'score')
top_av   = df_todas[df_todas['tipo_regra'] == 'aversao'].nlargest(10, 'score') if len(rules_aversao) > 0 else pd.DataFrame()

top50 = pd.concat([top_pref, top_av], ignore_index=True)
top50 = top50.nlargest(50, 'score').reset_index(drop=True)
top50.index = range(1, len(top50) + 1)
top50.index.name = 'rank'

# Colunas legíveis
top50 = top50.rename(columns={
    'support':       'suporte',
    'confidence':    'confianca',
    'leverage':      'leverage',
    'conviction':    'conviction',
    'zhangs_metric': 'zhang',
})
top50['suporte_pct']   = (top50['suporte'] * 100).round(2).astype(str) + '%'
top50['confianca_pct'] = (top50['confianca'] * 100).round(1).astype(str) + '%'
top50['lift']          = top50['lift'].round(3)
top50['score']         = top50['score'].round(4)

# Exportar — utf-8-sig é compatível com Excel no Windows
output_path = 'output/top50_regras_nao_obvias.csv'
top50.to_csv(output_path, index=True, encoding='utf-8-sig')

n_pref = (top50['tipo_regra'] == 'preferencia').sum()
n_av   = (top50['tipo_regra'] == 'aversao').sum()
print(f"\n✅ CSV exportado: {output_path}")
print(f"   {len(top50)} regras  |  {n_pref} preferência  |  {n_av} aversão")

# Preview
cols_preview = ['tipo_regra', 'Antecedente', 'Consequente', 'suporte_pct', 'confianca_pct', 'lift', 'score']
cols_preview = [c for c in cols_preview if c in top50.columns]
display(
    top50[cols_preview]
    .style
    .format({'lift': '{:.3f}', 'score': '{:.4f}'})
    .background_gradient(subset=['lift'], cmap='YlOrRd')
    .set_caption("Top 50 Regras de Associação Não-Óbvias")
)

---
## Conclusão

1. **Esparsidade (~95.5%)** exigiu suporte baixo para capturar padrões relevantes.
2. **Cauda longa** concentra avaliações em poucos filmes — viés de popularidade domina as regras.
3. **Binarização dupla:** nota ≥ 4 = preferência; nota ≤ 2 = aversão. Dois perfis de comportamento explorados.
4. **Filtro de regras óbvias:** sequências e episódios da mesma franquia removidos por sobreposição de palavras-chave nos títulos (≥60%), tornando as regras mais úteis para recomendação.
5. **FP-Growth** foi a escolha certa pelo tamanho e esparsidade (Apriori seria impraticável).
6. Para recomendação real, combinar regras de associação com filtragem colaborativa ou fatoração matricial.